## 1.1 Load and Inspect the Data

In [33]:
# Load the `ohlc_hw` table from `data/data.db`

import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("data/data.db")

df = pd.read_sql("SELECT * from ohlc_hw", conn)
df_calendar = pd.read_sql("SELECT * from ohlc", conn)
conn.close()

display(df.head())
display(df_calendar.head())

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,ts,open,high,low,close,adj_close,volume,ticker
0,2024-10-14,161.40,162.14,160.90,161.60,161.60,5964000,JNJ
1,2024-10-11,162.18,162.66,161.20,161.46,161.46,5253500,JNJ
2,2024-10-10,161.36,161.39,160.00,160.51,160.51,4793900,JNJ
3,2024-09-30,161.57,162.18,160.68,162.06,162.06,6010407,JNJ
4,2024-09-27,162.00,163.43,161.34,161.40,161.40,7553905,JNJ


In [34]:
# Display basic statistics (`.info()`, `.describe()`)

df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 24.3 MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


In [35]:
# Identify the columns and their meanings
# Copied from github lecture 2/hw2.md

print( "This table contains daily OHLC (Open, High, Low, Close) data for multiple stocks.")

print( "The columns are ts (observation date), ticker, raw open, high, low, close, vendor adj_close, and volume")


This table contains daily OHLC (Open, High, Low, Close) data for multiple stocks.
The columns are ts (observation date), ticker, raw open, high, low, close, vendor adj_close, and volume


In [36]:
# Show the date range and list of tickers

df['ts'] = pd.to_datetime(df['ts'])
ts_col = df['ts']

#date range
ts_col.min(), ts_col.max()

(Timestamp('1962-01-02 00:00:00'), Timestamp('2024-12-06 00:00:00'))

In [37]:
#list of tickers (sorted)
ticker_col = df['ticker']
sorted(ticker_col.unique())

['<a target="_blank"><span style="color: blue">AAPL</span></a>',
 '<a target="_blank"><span style="color: blue">AMGN</span></a>',
 '<a target="_blank"><span style="color: blue">AMZN</span></a>',
 '<a target="_blank"><span style="color: blue">AXP</span></a>',
 '<a target="_blank"><span style="color: blue">BA</span></a>',
 '<a target="_blank"><span style="color: blue">CAT</span></a>',
 '<a target="_blank"><span style="color: blue">CRM</span></a>',
 '<a target="_blank"><span style="color: blue">CSCO</span></a>',
 '<a target="_blank"><span style="color: blue">CVX</span></a>',
 '<a target="_blank"><span style="color: blue">DIS</span></a>',
 '<a target="_blank"><span style="color: blue">GS</span></a>',
 '<a target="_blank"><span style="color: blue">HD</span></a>',
 '<a target="_blank"><span style="color: blue">HON</span></a>',
 '<a target="_blank"><span style="color: blue">IBM</span></a>',
 '<a target="_blank"><span style="color: blue">JNJ</span></a>',
 '<a target="_blank"><span style="color

## 1.2 Detect Data Quality Issues

In [38]:
## 1.2 Detect Data Quality Issues
import html

RAW_PX = ['open', 'high', 'low', 'close']
ALL_PX = RAW_PX + ['adj_close']
VALUE_COLS = ALL_PX + ['volume']
KEY_COLS = ['ticker', 'ts']
OHLC_ORDER = [('low', 'open'), ('open', 'high'), ('low', 'close'), ('close', 'high'), ('low', 'high')]  # (smaller, larger)
REPORT_COLUMNS = ['ticker', 'issue_type', 'issue_date', 'description', 'severity']
ROW_SEVERITY = {'missing_values': 'low', 'validity': 'high', 'volume_zero': 'high'}


# ---- ticker / key normalization ----
def normalize_ticker(tickers):
    """Strip HTML markup/entities, whitespace and case: '<a><span>shw </span></a>' -> 'SHW'.
    Labels that end up empty become missing. Idempotent."""
    text = tickers.astype('string').str.replace(r'<[^>]*>', '', regex=True)
    text = text.map(html.unescape, na_action='ignore').astype('string')
    return text.str.replace(r'[\s\u200b\ufeff]+', '', regex=True).str.upper().replace('', pd.NA)


def ticker_changed(raw, clean):
    """True where a non-missing raw label differs from its normalized form."""
    same = (raw.astype('string') == clean).fillna(False).astype(bool)
    return raw.notna() & ~same


def ticker_normalization_audit(frame):
    """Distinct raw labels altered by normalization, with row counts."""
    clean = normalize_ticker(frame['ticker'])
    changed = ticker_changed(frame['ticker'], clean)
    return (pd.DataFrame({'ticker_raw': frame['ticker'][changed], 'ticker_clean': clean[changed]})
            .groupby(['ticker_raw', 'ticker_clean'], dropna=False).size().rename('n_rows').reset_index())


def normalize_keys(frame):
    """Copy of frame with normalized ticker and datetime ts (index preserved)."""
    return frame.assign(ticker=normalize_ticker(frame['ticker']), ts=pd.to_datetime(frame['ts'], errors='coerce'))


def _keyed_rows(frame):
    """Normalized rows that have a usable (ticker, ts) key."""
    return normalize_keys(frame).dropna(subset=KEY_COLS)


def _unique_rows(frame):
    """Keyed rows with exact duplicates (same normalized key and values) removed."""
    rows = _keyed_rows(frame)
    return rows[~rows.duplicated(KEY_COLS + VALUE_COLS)]


def _ticker_dates(frame):
    """Unique (normalized ticker, datetime) pairs."""
    return _keyed_rows(frame)[KEY_COLS].drop_duplicates()


# ---- row-level checks ----
def check_missing_values(row):
    return [("missing_values", f"{col} is missing") for col, value in row.items() if pd.isna(value)]


def check_validity(row):
    # Missing values are reported by check_missing_values, so they are skipped here.
    # adj_close only gets the positivity test; OHLC bounds apply to raw prices.
    issues = []
    for col in ALL_PX:
        if pd.notna(row[col]) and not (np.isfinite(row[col]) and row[col] > 0):
            issues.append(("validity", f"nonpositive or non-finite {col} price"))
    for smaller, larger in OHLC_ORDER:
        if pd.notna(row[smaller]) and pd.notna(row[larger]) and row[smaller] > row[larger]:
            issues.append(("validity", f"{smaller} > {larger} (violates low <= open/close <= high)"))
    return issues


def check_volume_zero(row):
    return [("volume_zero", "zero volume day (could indicate stale data)")] if row['volume'] == 0 else []


# ---- dataset-level checks ----
def check_duplicates(df):
    """Exact duplicates: same normalized key and values as an earlier row (safe to drop)."""
    rows = _keyed_rows(df)
    return rows.loc[rows.duplicated(KEY_COLS + VALUE_COLS), KEY_COLS]


def check_conflicting_keys(df):
    """(ticker, ts) keys that still hold several different rows once exact duplicates are removed."""
    rows = _unique_rows(df)
    clashes = rows[rows.duplicated(KEY_COLS, keep=False)]
    if clashes.empty:
        return pd.DataFrame(columns=KEY_COLS + ['n_rows', 'differing_columns'])
    by_key = clashes.groupby(KEY_COLS)
    differs = by_key[VALUE_COLS].nunique(dropna=False).gt(1)
    return pd.DataFrame({'n_rows': by_key.size(),
                         'differing_columns': differs.apply(lambda f: ', '.join(f.index[f.to_numpy()]), axis=1)}
                        ).reset_index()


def check_missing_trading_days(df, df_calendar):
    """One row per contiguous gap (in trading-day terms) inside each ticker's observed date range."""
    observed, calendar = _ticker_dates(df), _ticker_dates(df_calendar)
    gaps = []
    for ticker, seen in observed.groupby('ticker')['ts']:
        first, last = seen.min(), seen.max()
        days = calendar.loc[calendar['ticker'] == ticker, 'ts']
        days = days[days.between(first, last)].sort_values()
        missing = ~days.isin(seen)
        run_id = (missing != missing.shift(fill_value=False)).cumsum()  # consecutive missing days share an id
        for _, run in days[missing].groupby(run_id[missing]):
            gaps.append((ticker, run.iloc[0], run.iloc[-1], len(run), first, last))
    return pd.DataFrame(gaps, columns=['ticker', 'gap_start', 'gap_end', 'n_missing', 'ticker_start', 'ticker_end'])


def summarize_missing_trading_days(gaps):
    # tickers with no gaps are omitted
    for ticker, g in gaps.groupby('ticker'):
        print(f"Ticker {ticker} is missing {g['n_missing'].sum()} trading days between "
              f"{g['ticker_start'].iloc[0]:%Y-%m-%d} and {g['ticker_end'].iloc[0]:%Y-%m-%d}")


def check_non_trading_days(df, df_calendar):
    """Observed dates inside a ticker's calendar span that are not valid trading days."""
    calendar = _ticker_dates(df_calendar)
    span = calendar.groupby('ticker')['ts'].agg(['min', 'max'])
    merged = (_ticker_dates(df).join(span, on='ticker', how='inner')
              .merge(calendar.assign(is_valid=True), on=KEY_COLS, how='left'))
    outside = merged['is_valid'].isna() & merged['ts'].between(merged['min'], merged['max'])
    return merged.loc[outside, KEY_COLS]


def flag_return_outliers(frame, price_col='close', n_std=3.0):
    """Per-ticker daily return and 3-sigma flag, aligned to frame's (unique) index.
    Flag: abs(ret - mean) > n_std * sample std (ddof=1), both tails. Invalid prices and undefined
    returns are excluded; zero/undefined std never flags. A review flag, not a deletion rule."""
    prices = normalize_keys(frame[KEY_COLS + [price_col]])          # clean ticker before grouping
    prices = prices[np.isfinite(prices[price_col]) & (prices[price_col] > 0)].sort_values(KEY_COLS, kind='mergesort')
    ret = prices[price_col] / prices.groupby('ticker')[price_col].shift(1) - 1
    by_ticker = ret.groupby(prices['ticker'])
    deviation = ret - by_ticker.transform('mean')
    std = by_ticker.transform('std').where(lambda s: s > 0)  # zero/undefined std -> NaN -> never flagged
    screen = pd.DataFrame({'ret': ret, 'z': deviation / std,
                           'is_outlier': deviation.abs() > n_std * std}).reindex(frame.index)
    screen['is_outlier'] = screen['is_outlier'].eq(True)     # excluded rows -> False
    return screen


def check_outliers(df, n_std=3.0):
    rows = _unique_rows(df)   # repeated rows would create fake 0% returns
    screen = flag_return_outliers(rows, 'close', n_std)
    return screen[screen['is_outlier']].join(rows[KEY_COLS])[['ticker', 'ts', 'ret', 'z']]


# ---- report ----
def report_data_quality(df, df_calendar=None, n_std=3.0):
    """ticker | issue_type | issue_date | description | severity (high/medium/low)"""
    df = normalize_keys(df)   # clean ticker / datetime ts before any grouping
    issues = []
    for row in df.to_dict(orient='records'):
        for check in (check_missing_values, check_validity, check_volume_zero):
            for issue_type, description in check(row):
                severity = 'high' if description == 'volume is missing' else ROW_SEVERITY[issue_type]
                issues.append((row['ticker'], issue_type, row['ts'], description, severity))

    issues += [(r.ticker, 'duplicate_row', r.ts, 'exact duplicate (ticker, ts) row', 'low')
               for r in check_duplicates(df).itertuples()]
    issues += [(r.ticker, 'conflicting_key', r.ts,
                f"{r.n_rows} different rows share this (ticker, ts); differing columns: {r.differing_columns}",
                'high')
               for r in check_conflicting_keys(df).itertuples()]
    issues += [(r.ticker, 'outlier_return', r.ts,
                f"daily return {r.ret:.2%} is {abs(r.z):.1f} std from ticker mean (review flag, not deletion rule)",
                'high')
               for r in check_outliers(df, n_std).itertuples()]

    if df_calendar is not None:
        issues += [(r.ticker, 'missing_trading_days', r.gap_start,
                    f"Ticker {r.ticker} is missing {r.n_missing} trading days between "
                    f"{r.gap_start:%Y-%m-%d} and {r.gap_end:%Y-%m-%d}",
                    'medium')
                   for r in check_missing_trading_days(df, df_calendar).itertuples()]
        issues += [(r.ticker, 'non_trading_day', r.ts, 'date is not a valid trading day for this ticker', 'medium')
                   for r in check_non_trading_days(df, df_calendar).itertuples()]

    report = pd.DataFrame(issues, columns=REPORT_COLUMNS)
    return report.sort_values(['ticker', 'issue_date', 'issue_type']).reset_index(drop=True)


display(ticker_normalization_audit(df))
dq_report = report_data_quality(df, df_calendar)
display(dq_report.head(20))
display(dq_report.groupby(['issue_type', 'severity']).size().rename('count').reset_index())
summarize_missing_trading_days(check_missing_trading_days(df, df_calendar))

,ticker_raw,ticker_clean,n_rows
0,"<a target=""_blank""><span style=""color: blue"">A...",AAPL,307
1,"<a target=""_blank""><span style=""color: blue"">A...",AMGN,294
2,"<a target=""_blank""><span style=""color: blue"">A...",AMZN,185
3,"<a target=""_blank""><span style=""color: blue"">A...",AXP,356
4,"<a target=""_blank""><span style=""color: blue"">B...",BA,462
...,...,...,...
115,"<a target=""_blank""><span style=""font-weight: b...",TRV,371
116,"<a target=""_blank""><span style=""font-weight: b...",UNH,273
117,"<a target=""_blank""><span style=""font-weight: b...",V,96
118,"<a target=""_blank""><span style=""font-weight: b...",VZ,284


,ticker,issue_type,issue_date,description,severity
0,AAPL,outlier_return,1980-12-26,daily return 9.23% is 3.3 std from ticker mean...,high
1,AAPL,duplicate_row,1981-01-02,"exact duplicate (ticker, ts) row",low
2,AAPL,duplicate_row,1981-01-05,"exact duplicate (ticker, ts) row",low
3,AAPL,missing_values,1981-01-08,adj_close is missing,low
4,AAPL,missing_values,1981-01-13,adj_close is missing,low
5,AAPL,duplicate_row,1981-02-10,"exact duplicate (ticker, ts) row",low
6,AAPL,missing_values,1981-02-12,adj_close is missing,low
7,AAPL,duplicate_row,1981-02-27,"exact duplicate (ticker, ts) row",low
8,AAPL,duplicate_row,1981-03-05,"exact duplicate (ticker, ts) row",low
9,AAPL,duplicate_row,1981-03-06,"exact duplicate (ticker, ts) row",low


,issue_type,severity,count
0,duplicate_row,low,29460
1,missing_values,low,19855
2,outlier_return,high,5056
3,validity,high,10
4,volume_zero,high,81


## 1.3 Clean the Data

In [39]:
## 1.3 Clean the Data
"""
CLEANING POLICY
 1. Keys: normalize ticker (markup, whitespace, case) and ts (calendar date) BEFORE sorting, grouping or
    deduplicating. `ticker_raw` keeps the original label; `flag_ticker_normalized` marks changes.
    No usable (ticker, ts) -> quarantine.
 2. Sort by (ticker, ts). Exact duplicates (same key and values) are removed (first kept). Keys that still hold
    different rows cannot be resolved automatically -> all their rows are quarantined.
 3. Impossible values (nonpositive/non-finite price, negative/non-finite volume) -> NaN. Rows with all raw
    prices missing are removed. Missing volume -> 0 with `flag_volume_imputed` (unknown != no trading).
 4. Only CONFIRMED close errors are interpolated. Confirmed = extreme close return (3 std) that the next day
    reverses (a split would persist), on three consecutive trading days of the `ohlc` calendar with observed
    closes, and the neighbour mean lies inside the day's observed low/high. Replacement = mean of the two
    neighbouring closes in the sorted ticker; series ends are never repaired. Evidence goes to `repair_evidence`.
    Reversals lacking that corroboration are KEPT, flagged, and described as uncertain in `outlier_note`.
 5. Impossible OHLC relations (Option 3): rebuild high/low from the other prices (open/close anchor returns).
 6. Missing raw prices: forward-fill WITHIN ticker only, clamp filled open/close into the day's observed
    [low, high]. Leading gaps are never filled -> quarantined.
 7. adj_close = raw close x same-date factor (adj_close/close from `ohlc`; the row's own ratio only if there is no
    unambiguous calendar match and the close was not repaired). Factors are never carried across dates.
    Rebuilt where adj_close is missing or close changed. No trusted factor -> quarantined.
 8. After EVERY repair step: re-repair relations, re-derive adj_close, and record a recheck (`recheck_audit`).
 9. Remaining extreme returns are kept and flagged with the evidence or uncertainty; no split/news claims.
10. Missing trading days are reported in 1.2, never synthesized.
The raw table `ohlc_hw` is never modified; output goes to the NEW table `ohlc_hw_cleaned` (ts as YYYY-MM-DD).
"""

NOTE_TEMPLATES = {
    'confirmed': ("close {close:.4f} replaced by {replacement:.4f}, the mean of adjacent trading-day closes "
                  "{prev_close:.4f} and {next_close:.4f}. Evidence: extreme return {ret:+.1%} reversed the next day "
                  "(neighbour closes differ by {neighbour_move:+.1%}); {adj_note}; the day's observed low/high "
                  "contain the replacement."),
    'suspected': ("close return {ret:+.1%} (z={z:+.1f}) reverses next day, which suggests a bad print, but the evidence "
                  "is incomplete (needs three consecutive observed trading days and a day range containing the "
                  "replacement); not repaired, kept for review."),
    'split_like': ("close return {ret:+.1%} while adj_close return {adj_ret:+.1%} is not extreme: resembles a "
                   "corporate-action adjustment, but no split record confirms it; kept for review."),
    'unexplained': ("close return {ret:+.1%} (z={z:+.1f}) with no confirming reversal; cause not determined from the "
                    "data (no split or news evidence); not repaired, kept for review."),
}


def _ohlc_violation(frame):
    # NaN comparisons are False, so missing values never count as violations
    return pd.concat([frame[smaller] > frame[larger] for smaller, larger in OHLC_ORDER], axis=1).any(axis=1)


def _envelope_repair(frame, rows):
    """Option 3: rebuild high/low from all raw prices on the flagged rows."""
    if not rows.any():
        return
    for col, rebuilt in {'high': frame[RAW_PX].max(axis=1), 'low': frame[RAW_PX].min(axis=1)}.items():
        target = rows & frame[col].notna()
        frame.loc[target, col] = rebuilt[target]
    frame.loc[rows, 'flag_ohlc_repaired'] = True


def _adjustment_factors(df_calendar):
    """Trusted date-specific factor = adj_close / close from the supplied `ohlc` table.
    Keys with conflicting factors are dropped (ambiguous, so not trusted)."""
    cal = _keyed_rows(df_calendar[KEY_COLS + ['close', 'adj_close']])
    cal = cal.assign(close=pd.to_numeric(cal['close'], errors='coerce'),
                     adj_close=pd.to_numeric(cal['adj_close'], errors='coerce'))
    prices = cal[['close', 'adj_close']]
    cal = cal[(prices > 0).all(axis=1) & np.isfinite(prices).all(axis=1)]
    cal = cal.assign(calendar_factor=cal['adj_close'] / cal['close']).drop_duplicates(KEY_COLS + ['calendar_factor'])
    return cal.loc[~cal.duplicated(KEY_COLS, keep=False), KEY_COLS + ['calendar_factor']]


def _calendar_reference(df_calendar):
    """Per (ticker, ts) in `ohlc`: position in the ticker's trading calendar + trusted same-date factor."""
    days = _ticker_dates(df_calendar).sort_values(KEY_COLS)
    days['calendar_pos'] = days.groupby('ticker').cumcount()
    return days.merge(_adjustment_factors(df_calendar), on=KEY_COLS, how='left')


def _rebuild_adj(frame):
    """adj_close = close * same-date factor, where adj_close is missing or close was repaired."""
    needs_rebuild = frame['adj_close'].isna() | frame['close'].ne(frame['close_original'])
    rebuilt = frame['close'] * frame['adj_factor']      # NaN when no trusted factor exists
    frame.loc[needs_rebuild, 'adj_close'] = rebuilt[needs_rebuild]
    frame.loc[needs_rebuild & rebuilt.notna(), 'flag_adj_reconstructed'] = True


def _outlier_screens(frame, n_std):
    return flag_return_outliers(frame, 'close', n_std), flag_return_outliers(frame, 'adj_close', n_std)


def _recheck(frame, audit, stage):
    """Record what is still wrong: raw OHLC bounds, missing/invalid prices, adj_close != close x factor (rebuilt rows)."""
    prices = frame[ALL_PX]
    adj_matches = np.isclose(frame['adj_close'], frame['close'] * frame['adj_factor'], rtol=1e-9)
    audit.append({'stage': stage,
                  'raw_bound_violations': int(_ohlc_violation(frame).sum()),
                  'rows_missing_or_invalid_price': int((~((prices > 0) & np.isfinite(prices))).any(axis=1).sum()),
                  'adj_close_inconsistent_rows': int((frame['flag_adj_reconstructed'] & ~adj_matches).sum())})


def _restore_consistency(frame, audit, stage):
    """Run after EVERY repair: fix impossible OHLC relations, re-derive adj_close where close changed, recheck."""
    _envelope_repair(frame, _ohlc_violation(frame))
    _rebuild_adj(frame)
    _recheck(frame, audit, stage)


def _close_spike_evidence(frame, n_std, reversal_tol):
    """Judge one-day close spikes on observed closes inside each sorted ticker.
    confirmed: extreme return that the next day reverses, on three consecutive calendar trading days,
               with the neighbour mean inside the day's observed low/high.
    suspected: reverses but lacks that corroboration (never repaired)."""
    raw_screen, adj_screen = _outlier_screens(frame, n_std)
    by_ticker = frame.groupby('ticker')
    prev_close, next_close = by_ticker['close'].shift(1), by_ticker['close'].shift(-1)
    gap_before = frame['calendar_pos'] - by_ticker['calendar_pos'].shift(1)
    gap_after = by_ticker['calendar_pos'].shift(-1) - frame['calendar_pos']
    spike_move, neighbour_move = np.log(frame['close'] / prev_close), np.log(next_close / prev_close)
    reverts = raw_screen['is_outlier'] & (neighbour_move.abs() <= reversal_tol * spike_move.abs())
    replacement = (prev_close + next_close) / 2
    range_holds = (frame['low'] <= replacement) & (replacement <= frame['high'])
    status = np.select([reverts & (gap_before == 1) & (gap_after == 1) & range_holds, reverts],
                       ['confirmed', 'suspected'], default='')
    return pd.DataFrame({'status': status, 'close': frame['close'], 'replacement': replacement,
                         'prev_close': prev_close, 'next_close': next_close, 'ret': raw_screen['ret'],
                         'neighbour_move': np.exp(neighbour_move) - 1,
                         'adj_note': np.where(adj_screen['is_outlier'], 'adj_close also spiked', 'adj_close did not spike')},
                        index=frame.index)


def _notes(kinds, details):
    """Text per row: kinds picks the template ('' = no note), details supplies the numbers."""
    flagged = kinds != ''
    texts = [NOTE_TEMPLATES[kind].format(**row)
             for kind, row in zip(kinds[flagged], details[flagged].to_dict('records'))]
    return pd.Series(texts, index=kinds[flagged].index, dtype=object).reindex(kinds.index, fill_value='')


def _mark(fate, rows, status, reason):
    """Record why rows left the clean set."""
    ids = rows['raw_row_id'].to_numpy()
    fate.loc[ids, 'status'] = status
    fate.loc[ids, 'reason'] = reason


def clean_ohlc(raw, df_calendar, n_std=3.0, reversal_tol=0.25):
    source = raw.assign(raw_row_id=np.arange(len(raw)))
    fate = pd.DataFrame({'status': 'retained', 'reason': 'kept (repairs, if any, are flagged)'},
                        index=pd.RangeIndex(len(raw), name='raw_row_id'))
    log, recheck_audit = {}, []

    # normalize keys first; keep the original ticker label as an audit trail
    d = source.copy()
    d['ticker_raw'] = d['ticker']
    d['ticker'] = normalize_ticker(d['ticker'])
    d['flag_ticker_normalized'] = ticker_changed(d['ticker_raw'], d['ticker'])
    log['ticker_labels_normalized'] = int(d['flag_ticker_normalized'].sum())
    d['ts'] = pd.to_datetime(d['ts'], errors='coerce').dt.normalize()
    d[VALUE_COLS] = d[VALUE_COLS].apply(pd.to_numeric, errors='coerce')

    # rows without a key, exact duplicates, conflicting keys
    no_key = d[KEY_COLS].isna().any(axis=1)
    _mark(fate, d[no_key], 'quarantined', 'missing ts or ticker')
    d = d[~no_key].sort_values(KEY_COLS, kind='mergesort')
    exact_duplicate = d.duplicated(KEY_COLS + VALUE_COLS)
    _mark(fate, d[exact_duplicate], 'removed', 'exact duplicate of an earlier row')
    d = d[~exact_duplicate]
    conflicting = d.duplicated(KEY_COLS, keep=False)
    _mark(fate, d[conflicting], 'quarantined', 'conflicting rows share one (ticker, ts): investigate')
    d = d[~conflicting].reset_index(drop=True)

    # impossible values -> NaN, then drop rows with no raw price at all (before any filling)
    for col in VALUE_COLS:
        in_range = (d[col] >= 0) if col == 'volume' else (d[col] > 0)
        impossible = d[col].notna() & ~(in_range & np.isfinite(d[col]))
        log[f'invalid_{col}_set_to_nan'] = int(impossible.sum())
        d.loc[impossible, col] = np.nan
    all_prices_missing = d[RAW_PX].isna().all(axis=1)
    _mark(fate, d[all_prices_missing], 'removed', 'all raw prices missing')
    d = d[~all_prices_missing].reset_index(drop=True)

    # repair flags, originals, and the same-date reference (calendar position + trusted factor)
    for col in RAW_PX:
        d[f'flag_{col}_filled'] = d[col].isna()
    d['flag_volume_imputed'] = d['volume'].isna()
    for flag in ('flag_ohlc_repaired', 'flag_adj_reconstructed'):
        d[flag] = False
    d['close_original'] = d['close']
    d = d.merge(_calendar_reference(df_calendar), on=KEY_COLS, how='left')
    d['adj_factor'] = d['calendar_factor'].fillna(d['adj_close'] / d['close'])   # calendar first, own ratio as fallback
    d['volume'] = d['volume'].fillna(0)

    # repair 1: interpolate CONFIRMED close errors only (observed data, sorted ticker, valid neighbours)
    evidence = _close_spike_evidence(d, n_std, reversal_tol)
    confirmed = evidence['status'] == 'confirmed'
    d.loc[confirmed, 'close'] = evidence.loc[confirmed, 'replacement']
    d['flag_close_interpolated'] = confirmed
    d['repair_evidence'] = _notes(evidence['status'].where(confirmed, ''), evidence)
    d.loc[confirmed, 'adj_factor'] = d.loc[confirmed, 'calendar_factor']   # a bad close makes the row's own ratio untrustworthy
    log['close_errors_confirmed_and_interpolated'] = int(confirmed.sum())
    log['close_spikes_suspected_not_repaired'] = int((evidence['status'] == 'suspected').sum())
    _restore_consistency(d, recheck_audit, 'after close interpolation')

    # repair 2: forward-fill within ticker; filled open/close must sit inside the day's observed [low, high]
    d[RAW_PX] = d.groupby('ticker', sort=False)[RAW_PX].ffill()
    day_range_observed = ~d['flag_high_filled'] & ~d['flag_low_filled']
    for col in ('open', 'close'):
        filled = d[f'flag_{col}_filled'] & day_range_observed & d[col].notna()
        clamped = d[col].clip(lower=d['low'], upper=d['high'])
        d.loc[filled & clamped.ne(d[col]), 'flag_ohlc_repaired'] = True
        d.loc[filled, col] = clamped[filled]
    _restore_consistency(d, recheck_audit, 'after forward-fill and clamp')

    # rescreen repaired data: remaining extreme returns are kept, flagged, and described honestly
    raw_screen, adj_screen = _outlier_screens(d, n_std)
    raw_extreme = raw_screen['is_outlier']
    split_like = raw_extreme & ~adj_screen['is_outlier'] & adj_screen['ret'].notna()
    kind = pd.Series(np.select([raw_extreme & (evidence['status'] == 'suspected'), split_like, raw_extreme],
                               ['suspected', 'split_like', 'unexplained'], default=''), index=d.index)
    d['flag_outlier_review'] = raw_extreme
    d['outlier_note'] = _notes(kind, pd.DataFrame({'ret': raw_screen['ret'], 'z': raw_screen['z'],
                                                   'adj_ret': adj_screen['ret']}))
    log.update(kind[kind != ''].value_counts().add_prefix('extreme_returns_kept_').to_dict())

    # quarantine what cannot be repaired defensibly (never borrow from another ticker)
    prices_unresolved = d[RAW_PX].isna().any(axis=1)
    adj_unresolved = d['adj_close'].isna() & ~prices_unresolved
    _mark(fate, d[prices_unresolved], 'quarantined', 'unresolved leading gap: no prior price in this ticker')
    _mark(fate, d[adj_unresolved], 'quarantined', 'adj_close unresolved: no trusted same-date adjustment factor')
    d = d[~(prices_unresolved | adj_unresolved)]
    _recheck(d, recheck_audit, 'retained rows (final)')

    flag_cols = [c for c in d.columns if c.startswith('flag_')]
    df_clean = d[list(raw.columns) + ['ticker_raw', 'raw_row_id'] + flag_cols + ['repair_evidence', 'outlier_note']
                 ].reset_index(drop=True)
    df_clean['ticker'] = df_clean['ticker'].astype(object)       # plain str for SQLite

    # every input row has exactly one fate; quarantined rows keep their ORIGINAL values and labels
    retained_ids = fate.index[(fate['status'] == 'retained').to_numpy()].to_numpy()
    assert np.array_equal(np.sort(df_clean['raw_row_id'].to_numpy()), retained_ids)
    quarantined_ids = fate.index[(fate['status'] == 'quarantined').to_numpy()].to_numpy()
    df_quarantine = source.iloc[quarantined_ids].assign(quarantine_reason=fate.loc[quarantined_ids, 'reason'].to_numpy())
    return df_clean, df_quarantine, fate, pd.DataFrame(recheck_audit), log


def quality_stats(frame):
    """Counts for the before/after comparison."""
    keyed = normalize_keys(frame)
    prices = frame[ALL_PX].apply(pd.to_numeric, errors='coerce')
    volume = pd.to_numeric(frame['volume'], errors='coerce')
    return pd.Series({
        'rows': len(frame),
        'distinct ticker labels (as stored)': frame['ticker'].nunique(),
        'distinct tickers (normalized)': keyed['ticker'].nunique(),
        'duplicate (ticker, ts) keys': int(keyed.duplicated(KEY_COLS).sum()),
        'missing price cells': int(prices.isna().sum().sum()),
        'nonpositive/non-finite price cells': int((prices.notna() & ~((prices > 0) & np.isfinite(prices))).sum().sum()),
        'missing/invalid volume rows': int((~(np.isfinite(volume) & (volume >= 0))).sum()),
        'zero-volume rows': int((volume == 0).sum()),
        'OHLC relation violations': int(_ohlc_violation(prices).sum()),
        '3-std close-return outliers': len(check_outliers(frame)),
        'first date': f"{keyed['ts'].min():%Y-%m-%d}",
        'last date': f"{keyed['ts'].max():%Y-%m-%d}",
    })


def validate_clean(frame):
    """Hard validation of retained rows; raises AssertionError naming any failed check."""
    prices = frame[ALL_PX]
    checks = pd.Series({
        'raw + adjusted prices present, finite and positive': bool(((prices > 0) & np.isfinite(prices)).all().all()),
        '(ticker, ts) keys unique': not frame.duplicated(KEY_COLS).any(),
        'volume finite and nonnegative': bool((np.isfinite(frame['volume']) & (frame['volume'] >= 0)).all()),
        'low <= open/close <= high and low <= high': not _ohlc_violation(frame).any(),
    }, name='passed')
    assert checks.all(), f"validation failed:\n{checks[~checks]}"
    return checks


# ---- run ----
raw_fingerprint = pd.util.hash_pandas_object(df).sum()
df_clean, df_quarantine, row_fate, recheck_audit, cleaning_log = clean_ohlc(df, df_calendar)
assert pd.util.hash_pandas_object(df).sum() == raw_fingerprint       # in-memory raw data untouched

print("Cleaning log:")
for key, value in cleaning_log.items():
    print(f"  {key}: {value}")

# row accounting: retained / removed / quarantined
display(row_fate.groupby(['status', 'reason']).size().rename('rows').reset_index())
print(f"{len(df)} input rows = " + " + ".join(f"{n} {status}" for status, n in row_fate['status'].value_counts().items()))

# recheck after every repair, repair flags, and the evidence behind repairs / kept outliers
display(recheck_audit)
display(df_clean.filter(like='flag_').sum().rename('rows_flagged').to_frame())
display(df_clean.loc[df_clean['flag_ticker_normalized'], ['ticker_raw', 'ticker']].drop_duplicates().head())
display(df_clean.loc[df_clean['flag_close_interpolated'], ['ticker', 'ts', 'close', 'repair_evidence']].head())
display(df_clean.loc[df_clean['flag_outlier_review'], ['ticker', 'ts', 'close', 'outlier_note']].head())

# validation of retained rows
display(validate_clean(df_clean).to_frame())

# before / after statistics
display(pd.concat({'before (raw ohlc_hw)': quality_stats(df), 'after (cleaned)': quality_stats(df_clean)}, axis=1))
display(pd.concat({'before': df[['close', 'adj_close', 'volume']].describe(),
                   'after': df_clean[['close', 'adj_close', 'volume']].describe()}, axis=1).round(2))
display(pd.concat({'before': normalize_keys(df)['ticker'].value_counts(),
                   'after': df_clean['ticker'].value_counts()}, axis=1))

# detectors on the cleaned data. Expected leftovers: missing_trading_days (reported, not synthesized)
# and outlier_return (kept and flagged); no duplicate_row or conflicting_key rows should remain
post_report = report_data_quality(df_clean[df.columns], df_calendar)
display(post_report.groupby(['issue_type', 'severity']).size().rename('count').reset_index())

# Save df_clean as the NEW table ohlc_hw_cleaned in data/data.db (ts stored as ISO text YYYY-MM-DD)
conn = sqlite3.connect("data/data.db")
raw_rows_before = conn.execute("SELECT COUNT(*) FROM ohlc_hw").fetchone()[0]
df_clean.assign(ts=df_clean['ts'].dt.strftime('%Y-%m-%d')).to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)

# read back from SQL and re-verify what was actually stored
saved = pd.read_sql("SELECT * FROM ohlc_hw_cleaned", conn)
cutoff = '2000-01-01'
sql_rows_since_cutoff = conn.execute("SELECT COUNT(*) FROM ohlc_hw_cleaned WHERE ts >= ?", (cutoff,)).fetchone()[0]
raw_rows_after = conn.execute("SELECT COUNT(*) FROM ohlc_hw").fetchone()[0]
conn.close()

validate_clean(saved)
assert len(saved) == len(df_clean)
assert saved['ts'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all()                    # ISO dates only
assert sql_rows_since_cutoff == (df_clean['ts'] >= cutoff).sum()                # SQL date filter agrees with pandas
assert raw_rows_before == raw_rows_after == len(df)                             # raw table unchanged
print(f"Saved {len(saved)} rows to ohlc_hw_cleaned; ohlc_hw still has {raw_rows_after} rows.")

Cleaning log:
  ticker_labels_normalized: 39936
  invalid_open_set_to_nan: 5
  invalid_high_set_to_nan: 0
  invalid_low_set_to_nan: 0
  invalid_close_set_to_nan: 0
  invalid_adj_close_set_to_nan: 0
  invalid_volume_set_to_nan: 0
  close_errors_confirmed_and_interpolated: 116
  close_spikes_suspected_not_repaired: 133
  extreme_returns_kept_split_like: 4703
  extreme_returns_kept_suspected: 133
  extreme_returns_kept_unexplained: 92


,status,reason,rows
0,quarantined,unresolved leading gap: no prior price in this...,5
1,removed,exact duplicate of an earlier row,29460
2,retained,"kept (repairs, if any, are flagged)",368246


397711 input rows = 368246 retained + 29460 removed + 5 quarantined


,stage,raw_bound_violations,rows_missing_or_invalid_price,adj_close_inconsistent_rows
0,after close interpolation,0,5,0
1,after forward-fill and clamp,0,5,0
2,retained rows (final),0,0,0


,rows_flagged
flag_ticker_normalized,36985
flag_open_filled,0
flag_high_filled,0
flag_low_filled,0
flag_close_filled,0
flag_volume_imputed,0
flag_ohlc_repaired,0
flag_adj_reconstructed,18484
flag_close_interpolated,116
flag_outlier_review,4928


,ticker_raw,ticker
5,"<a target=""_blank""><span style=""color: red"">AA...",AAPL
10,"<a target=""_blank""><span style=""font-weight: b...",AAPL
36,"<a target=""_blank""><span style=""font-weight: b...",AAPL
213,"<a target=""_blank""><span style=""color: blue"">A...",AAPL
11063,"<a target=""_blank""><span style=""font-weight: b...",AMGN


... (4 outputs left out)

before                               after                         
           close  adj_close        volume      close  adj_close        volume
count  397711.00  377856.00  3.977110e+05  368246.00  368246.00  3.682460e+05
mean       46.29      51.65  3.097773e+07      46.32      51.00  3.095184e+07
std        67.14     194.58  1.266237e+08      67.15     190.63  1.266153e+08
min         0.01       0.00  0.000000e+00       0.01       0.00  0.000000e+00
25%         4.20       1.39  1.730700e+06       4.21       1.39  1.731514e+06
50%        19.86      11.96  4.648200e+06      19.91      11.96  4.649648e+06
75%        57.53      43.77  1.102380e+07      57.57      43.69  1.102160e+07
max       605.40   10442.51  9.276606e+09     605.40   10442.51  9.276606e+09

,before,after
ticker,,
DIS,17151,15867
KO,17147,15842
IBM,17138,15843
HON,17111,15808
MRK,17111,15841
PG,17108,15842
JNJ,17097,15807
CVX,17073,15842
CAT,17044,15808


,issue_type,severity,count
0,outlier_return,high,4928
1,volume_zero,high,74


Saved 368246 rows to ohlc_hw_cleaned; ohlc_hw still has 397711 rows.
